# 02 - Carga dos clientes no SQL Server
**Squad 2 - Grupo 2**

## Objetivo
Gravar os dados de `ecommerce_clientes` (lidos da Raw pelo notebook 01) na tabela `squad2.ecommerce_clientes` do SQL Server e conferir a carga lendo a tabela de volta.

## Regras e KPIs
Este notebook não gera regras de qualidade nem KPIs.

## Dependência
Executa o notebook 01 via `%run` para obter o `df_clientes`.

## Saídas
Tabela `squad2.ecommerce_clientes` no SQL Server. A gravação usa o modo `overwrite`, então a tabela é substituída por completo a cada execução.

## Execução do notebook 01
Roda o notebook de conexão com o ADLS para disponibilizar o `df_clientes`.

In [0]:
%run ./01_conexao_adls_gen2

## Importação das bibliotecas
Biblioteca para ler o `.env` e módulo `os`.

In [0]:
# Importação das bibliotecas necessárias

from dotenv import load_dotenv
import os

## Carregamento do `.env`
Localiza e carrega o `.env` com as credenciais do projeto.

In [0]:
# Localiza e carrega o arquivo .env com as credenciais do projeto

from pathlib import Path

env_path = next(Path("/Workspace").rglob(".env"), None)

if env_path:
    load_dotenv(env_path)
    print(".env carregado com sucesso.")
else:
    print(".env não encontrado.")

## Configurações do SQL Server
Lê host, banco, usuário e senha das variáveis de ambiente. Os valores não são exibidos.

In [0]:
# Lê as configurações do SQL Server a partir das variáveis de ambiente

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

print("Configurações do SQL Server carregadas.")

## URL JDBC
Monta a URL de conexão JDBC com o SQL Server (porta 1433, conexão criptografada).

In [0]:
# Monta a URL JDBC para conexão com o SQL Server

jdbc_url = (
    f"jdbc:sqlserver://{sql_host}:1433;"
    f"databaseName={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=true;"
)

print("URL JDBC preparada com sucesso.")

## Tabela de destino
Define `squad2.ecommerce_clientes` como destino da carga.

In [0]:
# Define a tabela de destino no SQL Server

tabela_destino = "squad2.ecommerce_clientes"

print(f"Tabela de destino: {tabela_destino}")

## Gravação no SQL Server
Grava o `df_clientes` na tabela de destino em modo `overwrite`.

In [0]:
# Grava os dados da ecommerce_clientes no SQL Server

df_clientes.write \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .option("encrypt", "true") \
    .option("trustServerCertificate", "true") \
    .mode("overwrite") \
    .save()

print("Dados carregados no SQL Server com sucesso.")

## Conferência da carga
Lê a tabela de volta do SQL Server e exibe os dados para validar a gravação.

In [0]:
# Lê a tabela de clientes do SQL Server

df_clientes_sql = spark.read \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .load()

display(df_clientes_sql)